# Corpus Inventory Review

Reproducible notebook for issue #22. It validates the approved August MVP corpus manifest, summarizes source/license/provenance fields, and writes then reads a review copy using the approved schema.

## Inputs

Set `SPACEBIO_INVENTORY_MANIFEST` to review a fixture or alternate manifest. Set `SPACEBIO_INVENTORY_REVIEW_OUTPUT` to choose where the schema-preserving review CSV is written.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from spacebio_evidence_engine.corpus import (
    load_inventory_manifest,
    load_inventory_review,
    summarize_inventory,
    write_inventory_manifest,
)

REPO_ROOT = Path.cwd()
manifest_path = Path(
    os.environ.get(
        "SPACEBIO_INVENTORY_MANIFEST",
        REPO_ROOT / "data" / "inventory" / "august_mvp_corpus_manifest.csv",
    )
)
review_output_path = Path(
    os.environ.get(
        "SPACEBIO_INVENTORY_REVIEW_OUTPUT",
        REPO_ROOT / "notebooks" / "generated" / "corpus_inventory_review.csv",
    )
)

manifest_path, review_output_path

## Load and validate manifest

In [ ]:
records = load_inventory_manifest(manifest_path)
summary = summarize_inventory(records)

summary

## Review source, license, and provenance coverage

In [ ]:
coverage = {
    "total_records": summary.total_records,
    "approved_records": summary.approved_records,
    "ingestible_records": summary.ingestible_records,
    "included_records": summary.included_records,
    "blocked_pdf_records": summary.blocked_pdf_records,
    "corpus_topics": summary.corpus_topics,
    "license_counts": summary.license_counts,
    "pdf_quality_counts": summary.pdf_quality_counts,
    "human_approval_counts": summary.human_approval_counts,
    "ingestion_status_counts": summary.ingestion_status_counts,
}

coverage

In [ ]:
review_rows = [
    {
        "publication_id": record.publication_id,
        "title": record.title,
        "doi": record.doi,
        "license": record.license,
        "license_status": record.license_status,
        "source_url": record.source_url,
        "pdf_url": record.pdf_url,
        "fulltext_url": record.fulltext_url,
        "pdf_quality": record.pdf_quality,
        "corpus_topic": record.corpus_topic,
        "organism_model": record.organism_model,
        "exposure": record.exposure,
        "ingestion_status": record.ingestion_status,
        "human_approval": record.human_approval,
    }
    for record in records
]

review_rows[:5]

## Write and read schema-preserving review copy

In [ ]:
write_inventory_manifest(records, review_output_path)
round_trip_records = load_inventory_review(review_output_path)

assert [record.publication_id for record in round_trip_records] == [
    record.publication_id for record in records
]
round_trip_sources = [record.source_url for record in round_trip_records]
round_trip_licenses = [record.license for record in round_trip_records]
assert round_trip_sources == [record.source_url for record in records]
assert round_trip_licenses == [record.license for record in records]

review_output_path